# Controlling MODFLOW 6 with the API

The MODFLOW 6 application programming interface (API) lets you drive a simulation from Python instead of running the `mf6` executable as a black box. Using the [`modflowapi`](https://github.com/MODFLOW-ORG/modflowapi) wrapper around the compiled MODFLOW 6 shared library, you can step through a model one time step (or even one solver iteration) at a time, read and modify internal variables while the model is running, and react to the simulated state.

The shared library is just MODFLOW 6 packaged differently: `libmf6` (with an OS-specific extension - `.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the stand-alone `mf6` executable are compiled from the **exact same source code**, so the numerical engine is identical and always in sync between them. The API does not reimplement or approximate anything - it simply exposes the MODFLOW 6 code you would run from the command line. **The take-home message: running a model through the API produces the same results as running it with the stand-alone executable** - the API only adds the ability to observe, steer, and override the simulation while it runs.

This notebook works through a series of increasingly complex examples: running a model through the API, running a coupled flow-and-transport model, monitoring solver convergence, changing inputs on the fly with a callback, and finally implementing a streamflow-augmentation rule that turns a well on only when simulated streamflow drops too low.

## How the API works: the big picture

Before diving into code, here is the mental model used throughout this notebook. Every example below is a variation on it, so it is worth a minute now.

**A MODFLOW 6 simulation is a nest of objects**

- a **simulation** contains one or more **models** — for example a groundwater-flow model (`GWF`) and a groundwater-transport model (`GWT`);
- each model contains **packages** — recharge (`RCH`), wells (`WEL`), streams (`SFR`), and so on;
- the models are advanced by one or more **solutions** (the numerical solvers), labelled `SLN_1`, `SLN_2`, ….

**Time is organized in three nested levels**

- a **stress period** is a block of time over which the boundary conditions (pumping, recharge, …) are held constant;
- each stress period is divided into one or more **time steps**;
- each time step is solved by repeating **outer (Picard) iterations** until the solution stops changing — i.e. it *converges*. The largest head change in an iteration is `HNCG`; the cap on the number of iterations is `MXITER`.

**Running a model through the API follows one lifecycle**

```text
mf6.initialize()                 # load the model into memory
while not finished:              # loop over time steps
    mf6.update()                 #   advance one time step (solves it internally)
mf6.finalize()                   # release memory and write the output files
```

`update()` hides the solver loop. When you need to reach *inside* a time step, you replace it with the manual loop:

```text
mf6.prepare_time_step(dt)
mf6.prepare_solve()
while not converged:             # outer iterations
    converged = mf6.solve()      #   one Picard iteration
mf6.finalize_solve()
mf6.finalize_time_step()
```

**Two ways to read and change data while the model runs**

- **High-level (callbacks):** `modflowapi` hands you the live simulation as familiar Python objects, so you can reach a package by name and edit its `stress_period_data` directly. This is the easiest way to work with the standard stress packages.
- **Low-level (direct memory access):** for variables the packages don't expose, you read MODFLOW's memory by *address* — a `(variable, model, package)` tag returned by `get_var_address`:
  - `get_value(tag)` returns a **copy**: a snapshot you can inspect without affecting the run, while
  - `get_value_ptr(tag)` returns a **pointer**: a live array that stays in sync with the running model and can be written back to change it.

Keep this picture — *simulation → models → packages*, the *initialize → update/solve → finalize* lifecycle, and *copy vs. pointer* — in mind as you go.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline
import flopy
import matplotlib.pyplot as plt
from modflowapi import ModflowApi, Callbacks, run_simulation
import numpy as np
import os
import pandas as pd
import pathlib as pl
import platform

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. Find the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the executable inside the active environment (here the project's pixi environment), and confirm that both exist.

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the mf6xtd pixi environment"

bin_path = "bin"
exe_ext = ""
if "linux" in platform.platform().lower():
    lib_ext = ".so"
elif "darwin" in platform.platform().lower() or "macos" in platform.platform().lower():
    lib_ext = ".dylib"
else:
    bin_path = "Scripts"
    lib_ext = ".dll"
    exe_ext = ".exe"
lib_name = env_path / f"{bin_path}/libmf6{lib_ext}"
if not lib_name.is_file():
    lib_name = env_path / f"lib/libmf6{lib_ext}"
if not lib_name.is_file():
    raise FileNotFoundError(
        f"Could not find mf6 library at in either: {env_path / 'bin'}"
        + f" or {env_path / 'lib'}"
    )

mf6_exe = env_path / f"{bin_path}/mf6{exe_ext}"

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Run a model with the API

The simplest use of the API: load an existing model, then drive it through time from Python. Instead of calling the `mf6` executable, initialize the library, repeatedly call `update()` to advance one time step until the end of the simulation is reached, and then finalize. This produces the same results as running the model with the executable.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
).resolve()
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api01")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()


In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")
    mf6.update()
    current_time = mf6.get_current_time()
mf6.finalize()

The loop above advanced the model one time step at a time and produced exactly the same results as running the `mf6` executable. Each `update()` call quietly ran the full solver loop for that time step — the next examples pry that loop open so you can act between iterations.

## Run a model with a manual solver loop

This example uses the watershed groundwater flow (GWF) model to show the manual solver loop. Instead of calling `update()`, it prepares each time step and then iterates the solver itself (`prepare_solve` / `solve` / `finalize_solve`) until the solution converges. Working at this level lets you read or modify the solution between outer iterations.

In [ ]:
ws = pl.Path(f"../data/watershed/gwf-only")
new_ws = pl.Path(f"models/watershed-gwf_api02")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter = mf6.get_value(mxit_tag)

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    kiter = 0
    mf6.prepare_solve()

    while kiter < max_iter:
        # here is where you could update well rates or other time-varying
        # inputs that depend on the solution from the previous iteration.
        # For example, you could get the current head solution and use it to
        # update a well rate based on a specified head difference and conductance.

        # solve with updated well rate
        has_converged = mf6.solve()
        kiter += 1

        if has_converged:
            msg = f"  Component {1}" + f" converged in {kiter}" + " outer iterations"
            print(msg)
            break

    if not has_converged:
        print("model did not converge")

    # finalize time step
    mf6.finalize_solve()

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()

## Run a coupled flow and transport model

Extend the manual solver loop to a simulation with more than one solution. The watershed model couples a groundwater flow (GWF) and a transport (GWT) model, so each time step loops over both solutions (`SLN_1` and `SLN_2`), solving each to convergence in turn.

In [ ]:
ws = pl.Path(f"../data/watershed/gwf-gwt")
new_ws = pl.Path(f"models/watershed-gwf-gwt_api03")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# maximum outer iterations
mxit_tag = mf6.get_var_address("MXITER", "SLN_1")
max_iter_gwf = mf6.get_value(mxit_tag)
mxit_tag = mf6.get_var_address("MXITER", "SLN_2")
max_iter_gwt = mf6.get_value(mxit_tag)


current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

while current_time < end_time:
    print(f"Running simulation time: {current_time}")

    # get dt and prepare for non-linear iterations
    dt = mf6.get_time_step()
    mf6.prepare_time_step(dt)

    # convergence loop
    for km in (1, 2):  # loop over gwf and gwt
        kiter = 0
        max_iter = max_iter_gwf if km == 1 else max_iter_gwt
        mf6.prepare_solve(component_id=km)

        while kiter < max_iter:
            # as in the previous example, you could update inputs here between
            # outer iterations - separately for the flow (km == 1) and transport
            # (km == 2) solutions
            if km == 1:
                pass  # update gwf inputs if needed
            else:
                pass  # update gwt inputs if needed

            # solve with updated well rate
            has_converged = mf6.solve(component_id=km)
            kiter += 1

            if has_converged:
                msg = (
                    f"  Component {km}" + f" converged in {kiter}" + " outer iterations"
                )
                print(msg)
                break

        if not has_converged:
            print("model did not converge")

        # finalize time step
        mf6.finalize_solve(component_id=km)

    # finalize time step and update time
    mf6.finalize_time_step()
    current_time = mf6.get_current_time()

print("Finalize simulation.")
mf6.finalize()

## Monitor convergence

Step into the solver to watch how the solution converges. After preparing each time step, loop over the solver's outer iterations manually and read the maximum head change (`HNCG`) at every iteration. The `|maximum head change|` for every iteration is plotted on a log scale and the figure is refreshed after each time step, so you can watch convergence happen **as the model runs**. The x-axis is grouped by stress period and time step (read from the `TDIS` package, with labels thinned when there are many); each time step is given an **equal-width slot** so the lengthy initial steady-state solve does not crowd out the equally interesting transient time steps. The figure uses the USGS publication style from `flopy.plot.styles`. This exposes the inner workings that are normally hidden when MODFLOW 6 runs on its own, and is useful for diagnosing slow or failing convergence.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
)
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.memory_print_option = "all"
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
from IPython.display import clear_output, display

mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# solver settings and the live pointer to the maximum head change per outer
# iteration (HNCG); KPER and KSTP are read from the TDIS package each time step
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
max_change = mf6.get_value_ptr(mf6.get_var_address("HNCG", "SLN_1"))
kper_tag = mf6.get_var_address("KPER", "TDIS")
kstp_tag = mf6.get_var_address("KSTP", "TDIS")

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

# convergence history, one entry per outer iteration:
# (cumulative iteration, |maximum head change|, stress period, time step)
history = []
not_converged = []
max_ticks = 25  # most SP/TS labels to show before thinning


def plot_convergence(fig, ax):
    ax.clear()
    if history:
        # group consecutive iterations into (stress period, time step) blocks
        blocks, b0 = [], 0
        for k in range(len(history)):
            if k == len(history) - 1 or history[k][2:] != history[k + 1][2:]:
                blocks.append((b0, k))
                b0 = k + 1

        # give every time step an equal-width slot on the x-axis so the long
        # initial steady-state solve (many outer iterations) does not crowd out
        # the transient time steps. Within slot bi (spanning [bi, bi + 1]) the
        # iterations are spread evenly, so each time step gets the same width
        # regardless of how many iterations it needed.
        xs, ys = [], []
        for bi, (i0, i1) in enumerate(blocks):
            n = i1 - i0 + 1
            for j, k in enumerate(range(i0, i1 + 1)):
                xs.append(bi + (j + 0.5) / n if n > 1 else bi + 0.5)
                yv = history[k][1]
                ys.append(np.nan if yv == 0.0 else yv)  # skip exact zeros on log axis
        ax.semilogy(
            xs,
            ys,
            marker="o",
            ms=4,
            lw=1.0,
            color="0.25",
            mfc="cyan",
            mec="0.25",
        )

        # thin the labels/separators so at most ~max_ticks are drawn
        stride = max(1, -(-len(blocks) // max_ticks))
        ticks, labels = [], []
        for bi, (i0, i1) in enumerate(blocks):
            if bi % stride == 0 or bi == len(blocks) - 1:
                ticks.append(bi + 0.5)
                labels.append(f"SP {history[i1][2]}\nTS {history[i1][3]}")
                if bi > 0:
                    ax.axvline(bi, color="0.5", lw=0.5)
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

    flopy.plot.styles.heading(ax=ax, heading="Outer iteration convergence")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period and time step")
    flopy.plot.styles.ylabel(ax=ax, label="Maximum head change, ft")
    flopy.plot.styles.remove_edge_ticks(ax=ax)

    clear_output(wait=True)
    display(fig)


with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)

    niter = 0
    while current_time < end_time:
        dt = mf6.get_time_step()
        mf6.prepare_time_step(dt)
        kper = int(mf6.get_value(kper_tag)[0])
        kstp = int(mf6.get_value(kstp_tag)[0])

        mf6.prepare_solve()
        kiter = 0
        has_converged = False
        while kiter < max_iter:
            has_converged = mf6.solve()
            niter += 1
            history.append((niter, abs(float(max_change[kiter])), kper, kstp))
            kiter += 1
            if has_converged:
                break

        plot_convergence(fig, ax)  # live update once per time step

        if not has_converged:
            not_converged.append((kper, kstp))

        mf6.finalize_solve()
        mf6.finalize_time_step()
        current_time = mf6.get_current_time()

    mf6.finalize()

plt.close(fig)  # the live figure is already displayed; avoid a duplicate

if not_converged:
    print("did not converge:", ", ".join(f"SP {p} TS {t}" for p, t in not_converged))
else:
    print("all time steps converged")

**How to read this plot.** Each marker is one outer (Picard) iteration. Within a time step the curve drops as the head change shrinks toward convergence, then resets at the start of the next time step — the vertical lines and `SP`/`TS` labels mark those breaks. A time step that reaches a small head change in just a few iterations converged easily; a curve that stays high or runs all the way to the `MXITER` cap flags a time step that is hard to solve. This is exactly the diagnostic information MODFLOW normally hides when it runs on its own.

## Increase precipitation with a callback

Modify model inputs while the simulation is running. The `callback_function` below is called by `modflowapi` at key points in the solution (for example `Callbacks.initialize`, `Callbacks.stress_period_start`, and `Callbacks.iteration_start`); here it intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation on the fly without rewriting the input files.

The callback is the most convenient way to work with the standard stress packages. `modflowapi` exposes the running model as familiar Python objects - the simulation (`sim`), each model (`sim.sv`), and its packages (`ml.rch_0`) - so you can reach a package by name and read or change its `stress_period_data` directly with numpy-style syntax (`spd["recharge"] *= 1.1`). The edits take effect in the running model immediately, and you never have to know the internal MODFLOW variable names, memory addresses, or array layout.

For data the package interface does not expose - internal solver variables or advanced-package terms, for example - you fall back to the lower-level `get_value` / `get_value_ptr` memory access introduced at the top of the notebook. The streamflow-augmentation example below uses it to read the simulated SFR flow and set the prediction-well rate.

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}"
)
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
def callback_function(sim, callback_step):
    """
    A demonstration function that dynamically adjusts recharge in a
    MODFLOW 6 model through the MODFLOW-API

    Parameters
    ----------
    sim : modflowapi.ApiSimulation
        A simulation object for the solution group that is
        currently being solved
    step : enumeration
        modflowapi.Callbacks enumeration object that indicates
        the part of the solution modflow is currently in.
    """
    ml = sim.sv
    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        rcha = ml.rch_0
        spd = rcha.stress_period_data
        print(f"updating recharge: stress_period={ml.kper + 1}")
        print(spd)
        spd["recharge"] *= 1.1

    if callback_step == Callbacks.timestep_start:
        pass

    if callback_step == Callbacks.iteration_start:
        # we can implement complex solutions to boundary conditions here!
        pass

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Confirm the callback changed the results

Running a callback is only convincing if you can see its effect. The cell below reads the **total global recharge** from the volumetric budget (`BUDGETCSV`) that the run above just wrote, and compares it with the original recharge.

There is no need to run the model a second time to get the baseline. The callback multiplies recharge by 1.1 in *every* stress period, so the scaling is uniform and the original recharge is simply the increased recharge divided by 1.1 - both curves come from this single run. (If a callback instead applied a *non-uniform* change, you would record the before/after values inside the callback itself, or compare against a separately saved baseline budget.) The two curves stay a constant 10% apart, which is the visual confirmation that the callback took effect.

In [ ]:
# read the total global recharge from the budget CSV the callback run just wrote.
# the callback scaled recharge by 1.1 uniformly, so the original recharge is just
# the increased recharge / 1.1 - no second model run is needed.
gwf = sim.get_model(name)
budget = gwf.output.budgetcsv().data

period = np.arange(1, len(budget) + 1)
rch_increased = budget["RCH(RCH_0)_IN"]
rch_baseline = rch_increased / 1.1

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(
        period,
        rch_baseline,
        marker="o",
        ms=4,
        lw=1.5,
        color="0.4",
        drawstyle="steps-mid",
        label="Original recharge",
    )
    ax.plot(
        period,
        rch_increased,
        marker="o",
        ms=4,
        lw=1.5,
        color="blue",
        drawstyle="steps-mid",
        label="Increased recharge (callback, +10%)",
    )
    flopy.plot.styles.heading(ax=ax, heading="Total global recharge")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge rate, ft$^3$/d")
    flopy.plot.styles.remove_edge_ticks(ax=ax)
    ax.legend(loc="best", frameon=False)
    fig.savefig(fig_path / "sv-recharge-callback.png", dpi=300, transparent=False)

## Augment streamflow with the prediction well

A more realistic application: use the API to implement a real-time operating rule. The goal is to keep simulated flow at the southern stream gage above a minimum threshold by pumping a prediction well only when needed. We first simulate a baseline scenario with the well turned off, then re-run with a callback that switches the well on whenever streamflow drops below the minimum, and finally compare the two.

### Simulate the baseline scenario

Run the advanced model with the prediction well turned off to establish the un-augmented streamflow at the southern gage. The head map shows the model boundaries (stream, pumping wells, and prediction well), and the streamflow is collected into a dataframe for later comparison.

> **A note on units.** This model's time unit is **days**, so MODFLOW reports stream and well flows in cubic feet per *day*. The cells below convert those to the conventional **cubic feet per second (cfs)** by dividing by `86400` (the number of seconds in a day), and multiply back by `86400` when handing a cfs rate to MODFLOW. The division is by `-86400` rather than `86400` simply to flip the sign into a positive-discharge convention. Separately, the value `1e30` that shows up in head arrays is MODFLOW's placeholder for dry or inactive cells; the map cell replaces it with a representative head so the contours plot cleanly.

In [ ]:
# run the base model
start_date = pd.to_datetime("1962-01-01 00:00:00")


Before processing any results, define two small helper functions used throughout the rest of the notebook:

- `sfr_gage_flows(gwf)` converts a run's SFR gage observations to cfs (the units note above), and
- `prediction_well_rate(gwf)` reads the prediction well's pumping rate from the budget - this one only matters later, in the streamflow-augmentation example.

Defining them once here keeps each result-processing cell short and identical from one scenario to the next; you do not need to study them now, just know they exist.

In [ ]:
def sfr_gage_flows(gwf):
    """SFR gage observations as a dataframe, converted from ft^3/day to cfs
    (and sign-flipped to a positive-discharge convention)."""
    df = gwf.sfr.output.obs().get_dataframe(start_datetime=start_date)
    df["RIV-FLOW"] /= -86400
    df["RIV-SWGW"] /= -86400
    df["TOTAL"] = df["RIV-SWGW"]
    Q0 = df["TOTAL"].iloc[0]
    df["PCT_DIFF"] = -100.0 * (df["TOTAL"] - Q0) / Q0
    return df


def prediction_well_rate(gwf):
    """Prediction-well (augmentation) rate per output time in cfs, read from
    the cell-by-cell budget; NaN when the well is off."""
    cobj = gwf.output.budget()
    rates = []
    for totim in cobj.get_times():
        rec = cobj.get_data(totim=totim, paknam2="prediction")[0]
        rates.append(0.0 if len(rec) == 0 else float(rec["q"][0]))
    rates = np.array(rates) / -86400.0
    rates[rates == 0.0] = np.nan
    return rates

In [ ]:
ws = pl.Path(
    f"../data/synthetic-valley/synthetic-valley-advanced-{sample_frequency}"    
)
new_ws = pl.Path(f"models/synthetic-valley-advanced-{sample_frequency}_api05")


sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=ws, exe_name=mf6_exe, write_headers=False
)
gwf = sim.get_model("sv")
pak = gwf.get_package("prediction")
new_spd = {}
for kper in range(sim.tdis.nper.array):
    spd = pak.stress_period_data.get_data(kper)
    if spd is None:
        continue
    spd["q"] = 0.0
    new_spd[kper] = spd
pak.stress_period_data.set_data(new_spd)

sim.set_sim_path(new_ws)
sim.write_simulation(silent=True)
sim.run_simulation(silent=True)

In [ ]:
aspect_ratio = 25.0 / 40.0
height = 6.6
width = aspect_ratio * height
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(width, height),
        sharex=True,
        constrained_layout=True,
    )
    mv = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax, extent=gwf.modelgrid.extent)
    hds = gwf.output.head().get_data()
    hds[hds == 1e30] = 14.066030871906671
    mv.plot_array(hds)
    CS = mv.contour_array(hds, levels=np.arange(1, 14, 1), colors="black")
    ax.clabel(CS, CS.levels, fmt="%.0f", fontsize=10)
    mv.plot_bc("SFR", color="green")
    mv.plot_bc(
        "WEL",
        package=gwf.pwell,
        plotAll=True,
        color="red",
        kper=12,
    )
    mv.plot_bc("WEL", package=gwf.prediction, plotAll=True, color="orange", kper=12)
    mv.plot_grid(lw=0.5, color="black")

    ax.set_xticklabels([])
    ax.set_yticklabels([])

    fig.savefig(fig_path / "sv-head.png", dpi=300, transparent=False)

In [ ]:
base_df = sfr_gage_flows(gwf)


In [ ]:
new_row = {
    start_date: {
        "totim": 0.0,
        "RIV-FLOW": np.nan,
        "RIV-SWGW": np.nan,
        "TOTAL": np.nan,
        "PCT_DIFF": np.nan,
        "augmentation rate": np.nan,
    }
}
new_row = pd.DataFrame(new_row).T
base_df = pd.concat([new_row, base_df])
base_df

### Define the streamflow-augmentation rule

Set the minimum acceptable streamflow and write the callback that enforces it. When the downstream SFR flow falls below `min_flow`, the prediction well is turned on at a rate proportional to the shortfall (capped at a maximum), and that water is added as stream inflow at the upstream end of STrainght River.

The callback always sets the rate at the start of each stress period using the streamflow from the end of the previous timestep. Setting `update_each_iteration = True` additionally recomputes the rate at the start of every outer iteration (`Callbacks.iteration_start`), so the augmentation tracks the latest simulated streamflow and converges with the rest of the solution instead of lagging it by one timestep.

To keep the per-iteration update stable, the rate is *under-relaxed*: each outer iteration it moves only a fraction `relax` of the way toward the rate the current streamflow calls for. This matters because pumping the prediction well also draws water from the stream, which causes more leakage from the stream and decreases streamflow; under-relaxation lets the rate stabilize and lets the pump switch back off when streamflow recovers.

In [ ]:
min_flow = 15.0

# the augmentation period begins after this many stress periods (depends on
# whether the model uses monthly or annual stress periods)
aug_start = 120 if sample_frequency == "monthly" else 10

# when True, the augmentation rate is also recomputed every outer iteration
# (Callbacks.iteration_start) from the latest streamflow; when False, it is set
# once per stress period from the previous timestep's streamflow
update_each_iteration = False

In [ ]:
# run the advanced model in the augmentation workspace through the API with the
# streamflow-augmentation callback defined below (ws0 is populated separately)
sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=new_ws, write_headers=False)

In [ ]:
# the prediction well's augmentation rate (ft^3/d). It persists across outer
# iterations so the per-iteration update can under-relax toward a converged value
# instead of ratcheting up to the cap.
aug_rate = 0.0

# under-relaxation factor for the per-iteration update (0 < relax <= 1); smaller
# values damp the change in rate between successive outer iterations
relax = 0.5


def callback_function(sim, callback_step):
    """Augment streamflow with the prediction well, following the rule described
    in the markdown above. `sim` is the running ApiSimulation and `callback_step`
    is the Callbacks stage modflowapi is currently in."""
    global aug_rate

    ml = sim.sv

    def required_rate():
        """Prediction-well rate (ft^3/d) the current streamflow calls for:
        proportional to the shortfall below min_flow (capped), and zero once the
        downstream SFR flow is at or above the target."""
        q = float(sim.mf6.get_value(sim.mf6.get_var_address("DSFLOW", "SV", "SFR-1"))[-1])
        q /= 86400.0
        if q < min_flow:
            return min(2.5 * (min_flow - q) * 86400.0, 2e6)
        return 0.0

    def apply_rate(rate):
        """Pump the prediction well at `rate` and add the same rate as stream inflow."""
        ml.prediction.stress_period_data["q"] = -rate
        inflow = sim.mf6.get_value_ptr(sim.mf6.get_var_address("INFLOW", "SV", "SFR-1"))
        inflow[0] = rate

    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        # seed the rate once per stress period from the previous timestep's flow
        if sim.kper > aug_start:
            aug_rate = required_rate()
            apply_rate(aug_rate)

    if callback_step == Callbacks.iteration_start:
        # recompute every outer iteration, under-relaxing toward the required rate
        # so it converges with the solution (the rate may rise OR fall) rather than
        # ratcheting up to the cap
        if update_each_iteration and sim.kper > aug_start:
            aug_rate += relax * (required_rate() - aug_rate)
            aug_rate = min(max(aug_rate, 0.0), 2e6)
            apply_rate(aug_rate)

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Compare baseline and augmented streamflow

Collect the augmented streamflow and the well augmentation rate from the model output, then plot the baseline and augmented flows together with the augmentation period and the minimum-flow threshold to see how much the operating rule raises streamflow toward the target.

In [ ]:
gwf = sim.get_model("SV")
df = sfr_gage_flows(gwf)
df

In [ ]:
prediction_rate = prediction_well_rate(gwf)

In [ ]:
df["augmentation rate"] = prediction_rate

In [ ]:
new_row = {
    start_date: {
        "totim": 0.0,
        "RIV-FLOW": np.nan,
        "RIV-SWGW": np.nan,
        "TOTAL": np.nan,
        "PCT_DIFF": np.nan,
        "augmentation rate": np.nan,
    }
}
new_row = pd.DataFrame(new_row).T
new_row

In [ ]:
df = pd.concat([new_row, df])
df

In [ ]:
x0 = df.index[aug_start]
x1 = df.index[-1]
xstart = df.index[1]
xend = df.index[-1]

In [ ]:
base_color, aug_color = "green", "blue"
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2,
        1,
        figsize=(12.3, 4.3),
        sharex=True,
        constrained_layout=True,
    )

    fig.suptitle("Southern Boundary - Gage 1")

    ax = axs[0]
    y0, y1 = 5, 30
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=base_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Base",
    )
    df["RIV-FLOW"][xstart:xend].plot(
        ax=ax,
        color=aug_color,
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.axhline(0, lw=0.5, color="black")
    ax.axhline(min_flow, lw=1.0, ls=":", color="blue", label="Minimum flow")
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("River\nDischarge, cfs")
    leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0 = 0
    if sample_frequency == "annual":
        y1 = 10
    else:
        y1 = 25

    area_df = df["augmentation rate"][xstart:xend].copy()

    area_df.plot(
        ax=ax,
        color="red",
        lw=1.5,
        ls="-",
        # clip_on=False,
        drawstyle=drawstyle,
        label="Augmented flow",
    )
    ax.fill_between(
        [x0, x1],
        y0,
        y1,
        color="cyan",
        alpha=0.25,
        lw=0,
        label="Augmentation period",
    )

    for idx in range(1, len(df.index)):
        xa0, xa1 = df.index[idx - 1], df.index[idx]
        ax.fill_between(
            [xa0, xa1],
            0,
            df["augmentation rate"].values[idx],
            color="red",
            lw=1.0,
            edgecolor="black",
            zorder=100,
        )
    # ax.set_xlim(xstart, xend)
    ax.set_ylim(y0, y1)

    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    # leg = ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_southern_boundary.png", dpi=300, transparent=False
    )

**What to look for.** During the shaded augmentation period the blue *Augmented* flow sits above the green *Base* (well-off) flow — the rule is adding water — and the bottom panel shows the prediction-well rate the callback chose: zero until the stream needs help, then rising with the shortfall. Notice, though, that the augmented flow still dips below the dotted `min_flow` line in the driest periods; it does not reach the target, and that is physical rather than a bug. The prediction well is hydraulically connected to the stream, so pumping it pulls water back out of the streambed and claws back most of the augmentation — in test runs the well lifted the gage only from about 12.3 to 13.3 cfs against the 15 cfs target. The rule supports the flow; at this well location it cannot by itself hold the line.

### Compare the two update strategies

Run the augmentation model twice - once recomputing the rate only at the start of each stress period (`update_each_iteration = False`) and once recomputing it every outer iteration (`update_each_iteration = True`) - and store each result in its own dataframe (`df_period` and `df_iter`). Plotting them together with the baseline shows how recomputing the rate every outer iteration tracks the current streamflow, while updating only once per stress period lags it by a period.

In [ ]:
# run the augmentation model with both update strategies, saving each run to its
# own dataframe (the baseline run is already stored in base_df)
results = {}
for label, flag in (("per stress period", False), ("per iteration", True)):
    update_each_iteration = flag
    run_simulation(lib_name, new_ws, callback_function, verbose=False)

    gwf = sim.get_model("SV")
    rdf = sfr_gage_flows(gwf)
    rdf["augmentation rate"] = prediction_well_rate(gwf)

    # prepend a start-date row so the series begins at the simulation start
    start_row = pd.DataFrame({start_date: {c: np.nan for c in rdf.columns}}).T
    start_row["totim"] = 0.0
    results[label] = pd.concat([start_row, rdf])

# the two strategies are kept in separate dataframes
df_period = results["per stress period"]
df_iter = results["per iteration"]
df_iter

In [ ]:
idx_ref = df_iter.index
x0, x1 = idx_ref[aug_start], idx_ref[-1]
xstart, xend = idx_ref[1], idx_ref[-1]
drawstyle = "steps-pre"

with flopy.plot.styles.USGSPlot():
    fig, axs = plt.subplots(
        2, 1, figsize=(12.3, 4.3), sharex=True, constrained_layout=True
    )
    fig.suptitle("Southern Boundary - Gage 1: update-strategy comparison")

    ax = axs[0]
    ax.set_ylim(5, 30)
    base_df["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="green", lw=1.5, drawstyle=drawstyle, label="Base"
    )
    df_period["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="orange", lw=1.5, drawstyle=drawstyle,
        label="Augmented (per stress period)",
    )
    df_iter["RIV-FLOW"][xstart:xend].plot(
        ax=ax, color="blue", lw=1.5, drawstyle=drawstyle,
        label="Augmented (per iteration)",
    )
    ax.axhline(min_flow, lw=1.0, ls=":", color="black", label="Minimum flow")
    ax.fill_between(
        [x0, x1], 5, 30, color="cyan", alpha=0.25, lw=0, label="Augmentation period"
    )
    ax.set_ylabel("River\nDischarge, cfs")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    ax = axs[1]
    y0 = 0
    if sample_frequency == "annual":
        y1 = 10
    else:
        y1 = 25
    ax.set_ylim(y0, y1)
    df_period["augmentation rate"][xstart:xend].plot(
        ax=ax, color="orange", lw=1.5, drawstyle=drawstyle, label="per stress period"
    )
    df_iter["augmentation rate"][xstart:xend].plot(
        ax=ax, color="blue", lw=1.5, drawstyle=drawstyle, label="per iteration"
    )
    ax.fill_between([x0, x1], y0, y1, color="cyan", alpha=0.25, lw=0)
    ax.set_ylabel("Augmentation\nrate, cfs")
    ax.set_xlabel("Year")
    ax.legend(loc="upper left", ncol=2, frameon=False)

    fig.align_ylabels()
    fig.savefig(
        fig_path / "sv_sfr_api_update_comparison.png", dpi=300, transparent=False
    )

**What to look for.** Recomputing the rate only *per stress period* (orange) sets it from the previous period's flow, so the pump reacts a step late - it can keep pumping into a period that has already recovered, or stay off in one that has just dropped. Recomputing it *per iteration* (blue) under-relaxes the rate toward the current shortfall as the solution converges, so the pump tracks the present streamflow - easing off as flow climbs back toward `min_flow` and ramping up when it falls. The summary table below compares the resulting gage flow and pumped volume.

In [ ]:
# quick numeric summary over the (shaded) augmentation period


def summarize(df, label):
    rate = df["augmentation rate"].fillna(0.0)  # cfs, 0 when the well is off
    dt_s = df["totim"].diff().fillna(0.0) * 86400.0  # period length, seconds
    aug = df.iloc[aug_start:]  # rows within the augmentation period
    flow = aug["RIV-FLOW"]
    return pd.Series(
        {
            "min gage flow (cfs)": flow.min(),
            "mean gage flow (cfs)": flow.mean(),
            "periods below target": int((flow < min_flow).sum()),
            "max augmentation (cfs)": rate.iloc[aug_start:].max(),
            "total augmented volume (acre-ft)": (rate * dt_s).iloc[aug_start:].sum()
            / 43560.0,
        },
        name=label,
    )


summary = pd.DataFrame(
    [
        summarize(base_df, "base"),
        summarize(df_period, "per stress period"),
        summarize(df_iter, "per iteration"),
    ]
)
summary.round(2)

The table puts numbers on the difference: the per-iteration rule responds to the present streamflow rather than the previous period's. The stream is leaky and some of the groundwater pumped returns to the groundwater from the stream - so neither strategy erases the deficit entirely.

**Recap.** Across these examples you have driven MODFLOW 6 from Python at every level — running a model through `update()`, opening the solver loop by hand, coupling flow and transport, watching convergence live, editing inputs through high-level package objects, and reading and writing internal memory to implement a real-time operating rule. The same lifecycle (`initialize → update`/`solve → finalize`) and the same two access patterns (package objects vs. `get_value` / `get_value_ptr`) underlie every one of them.